# Pandera

This module covers pandera, the runtime schema validator for pandas
DataFrames. Readers are assumed to know pandas at the level of the previous
module and to have a planning or analytical workload in mind, typically
reading from and writing to a TM1 cube. The focus here is the patterns that
turn "the data is probably right" into "the data is checked at the
boundary, and any failure is a structured exception with column, check, and
row attached."

Pandera occupies a specific niche. Ad hoc `assert` statements and
`df.dtypes` checks are easy to write but impossible to summarize. Heavier
data quality platforms such as Great Expectations cover more ground but add
configuration and ceremony. Pandera sits in between: a schema is an
ordinary Python object, validation is one call, errors are structured, and
the same schema doubles as living documentation of what a function expects.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later. The same running example, the
small sales-plan DataFrame introduced in the pandas module, threads through
every topic, so each section introduces exactly one new validation pattern.

---

## Topic list

1. Pandera in the validation landscape
2. The running example
3. DataFrameSchema and Column
4. Built-in checks
5. Custom checks
6. Coercion and dtype enforcement
7. Nullable, required, and strict mode
8. Index and MultiIndex schemas
9. Lazy validation: collecting every failure
10. Class-based schemas with DataFrameModel
11. Validating function inputs and outputs
12. Validating against TM1 dimension members
13. Synthetic data for tests
14. Real-world design principles
15. Common mistakes

---

## 1. Pandera in the validation landscape

Validation is the step that converts "the upstream system probably gave the
right shape" into "the upstream system gave the right shape, and if not,
the failure is named and located." For data flowing into and out of a TM1
cube, the failure modes are familiar. A region recorded as `"EMEA"`
instead of `"Europe"` after a manual edit. A revenue exported through Excel
and arriving as the string `"120000.00"` rather than a float. A `period`
column quietly dropped by an inner join. A units count posted as a negative
number because someone reversed a signed convention. Each of these is
silent without validation; each becomes a written, located exception with
pandera.

Pandera's central object is the schema. A schema names columns, declares
dtypes, lists allowed values, and attaches arbitrary checks. Calling
`schema.validate(df)` either returns the DataFrame unchanged or raises a
structured error pointing at the offending column, check, and row. Beyond
runtime validation, the schema can coerce dtypes (parse strings as floats),
generate example DataFrames for tests, and decorate functions so inputs and
outputs are validated automatically.

The standard import in current pandera is `import pandera.pandas as pa`,
which makes the pandas backend explicit; the shorter `import pandera as pa`
still works for the pandas backend and is what most existing code reads.
The two forms are used interchangeably below. The polars backend lives
behind `import pandera.polars` and is out of scope for this module.

Where pandera fits among neighbours: pydantic validates Python objects but
does not natively understand the columnar structure of a DataFrame; mypy
enforces types statically and never sees runtime data; Great Expectations
covers data documentation, profiles, and suites but is heavier than a
schema object. Pandera is the right tool when the workflow is "read a
DataFrame, transform it, write a DataFrame," and the cost of a silent
shape error reaches as far as a published TM1 cube.

## 2. The running example

The same sales-plan DataFrame from the pandas module is used throughout
this one. It represents the result of reading a small slice of a `Sales
Plan` cube via `tm1.cells.execute_view_dataframe`, with one row per
(period, region, product) tuple and two measures expressed as columns.

In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "period":  ["2026Q1", "2026Q1", "2026Q1", "2026Q2", "2026Q2", "2026Q2"],
    "region":  ["Europe", "Americas", "Asia", "Europe", "Americas", "Asia"],
    "product": ["Standard", "Standard", "Standard", "Premium", "Premium", "Premium"],
    "revenue": [120_000.0, 250_000.0, 90_000.0, 135_000.0, 270_000.0, 110_000.0],
    "units":   [1200, 2500, 900, 540, 1080, 440],
})

The shape is what matters; real plans have many more rows, more periods,
and several more measures. Throughout this module, the validation rules to
enforce on this frame are:

- `period` matches a fiscal quarter pattern (`YYYYQn`).
- `region` belongs to the known set of TM1 dimension elements.
- `product` belongs to the known product set.
- `revenue` is non negative and finite.
- `units` is a non negative integer.
- No extra columns sneak in from upstream.

Each of the topics below realizes one piece of this set, building the
schema up incrementally rather than presenting it whole.

## 3. DataFrameSchema and Column

The simplest schema names each column and declares its dtype. Validating
an unmodified `sales` against it returns the frame unchanged.

In [ ]:
import pandera as pa

sales_schema = pa.DataFrameSchema({
    "period":  pa.Column(str),
    "region":  pa.Column(str),
    "product": pa.Column(str),
    "revenue": pa.Column(float),
    "units":   pa.Column(int),
})

sales_schema.validate(sales)
# returns the same DataFrame, unchanged

The first failure mode caught by even this minimal schema is dtype drift.
A `revenue` column that arrived as `object` because one row carried a
string is rejected immediately, with the column name in the message.

In [ ]:
broken = sales.copy()
broken.loc[0, "revenue"] = "120000.00"     # arrived as string from Excel
sales_schema.validate(broken)
# pandera.errors.SchemaError: expected series 'revenue' to have type float64,
# got object

`Column` is the workhorse. Its constructor accepts a dtype, a list of
checks, nullability, requiredness, and a coercion flag, all introduced in
the next sections. Every behaviour is a keyword argument on `Column` or a
flag on the surrounding schema, so the schema reads top to bottom as a
single declaration rather than a chain of `.with_x(...)` calls.

A schema is non destructive. Validation never mutates the input. With
`coerce=True` (Topic 6), `validate` returns a new DataFrame with adjusted
dtypes; without it, the same object is returned by identity.

## 4. Built-in checks

A dtype check catches the wrong _type_ but not the wrong _value_. The next
layer is a check on the values themselves. Pandera ships built-ins for the
common cases.

In [ ]:
import pandera as pa

sales_schema = pa.DataFrameSchema({
    "period":  pa.Column(str, pa.Check.str_matches(r"^\d{4}Q[1-4]$")),
    "region":  pa.Column(str, pa.Check.isin({"Europe", "Americas", "Asia"})),
    "product": pa.Column(str, pa.Check.isin({"Standard", "Premium"})),
    "revenue": pa.Column(float, pa.Check.ge(0)),
    "units":   pa.Column(int, pa.Check.in_range(0, 1_000_000)),
})

sales_schema.validate(sales)

Each `pa.Check.X` is shorthand for a common predicate, applied element-wise
to the column. The full set covers comparison (`eq`, `ne`, `gt`, `ge`,
`lt`, `le`), range (`in_range`, `between`), set membership (`isin`,
`notin`), string patterns (`str_matches`, `str_contains`, `str_startswith`,
`str_endswith`, `str_length`), and uniqueness (`unique_values_eq`).

When a value violates a check, the error names the column, the check, and
the failing index.

In [ ]:
broken = sales.copy()
broken.loc[0, "region"] = "EMEA"
sales_schema.validate(broken)
# pandera.errors.SchemaError: Column 'region' failed element-wise validator 0:
# <Check isin: isin({'Americas', 'Asia', 'Europe'})>
# failure cases:
#    index failure_case
# 0      0         EMEA

Multiple checks on a single column are passed as a list. They run in
order, but every one is evaluated; pandera does not short circuit on the
first failure within a column unless eager validation surfaces it (Topic 9).

In [ ]:
pa.Column(int, [pa.Check.ge(0), pa.Check.le(1_000_000)])

Built-in checks cover the easy 80%. The remaining 20%, anything
domain specific, uses custom checks.

## 5. Custom checks

A custom check is any callable returning a boolean Series (or a single bool
for dataframe-level checks). The most common shape is element-wise: a
function `T -> bool` that pandera applies to each value.

A typical TM1 case is a domain rule that does not reduce to a built-in.
Suppose every Premium product must have a unit price of at least 200,
expressed as `revenue / units >= 200`. The rule depends on two columns, so
it lives on the DataFrame, not on a single column.

In [ ]:
def premium_min_unit_price(df: pd.DataFrame) -> pd.Series:
    return (df["product"] != "Premium") | (df["revenue"] / df["units"] >= 200)

sales_schema = pa.DataFrameSchema(
    columns={
        "period":  pa.Column(str, pa.Check.str_matches(r"^\d{4}Q[1-4]$")),
        "region":  pa.Column(str, pa.Check.isin({"Europe", "Americas", "Asia"})),
        "product": pa.Column(str, pa.Check.isin({"Standard", "Premium"})),
        "revenue": pa.Column(float, pa.Check.ge(0)),
        "units":   pa.Column(int, pa.Check.ge(0)),
    },
    checks=[
        pa.Check(premium_min_unit_price, name="premium_min_unit_price"),
    ],
)

Element-wise checks live on a column and receive the per-row value.
Dataframe-level checks live in the top-level `checks=` parameter and
receive the whole frame, returning a boolean Series with one entry per row
(or a single bool to flag the whole frame).

In [ ]:
pa.Column(float, pa.Check(lambda x: x % 1 == 0, name="whole_amounts"))
# element-wise: lambda receives a single value, returns bool

pa.Check(lambda df: df["revenue"].sum() < 10_000_000, name="total_cap")
# dataframe-level: lambda receives whole df, returns single bool

Naming the check via `name=` is worth the keystrokes. Pandera reports
failures by check name; an anonymous lambda surfaces as `<Check
_check_fn>`, which is no help when ten checks fail at once.

## 6. Coercion and dtype enforcement

A schema that rejects an `object` column for being the wrong dtype is
correct but inconvenient when the upstream source is known to be sloppy.
CSV exports, Excel sheets, and ad hoc TM1 view extracts often deliver
numeric columns as strings. Pandera's `coerce=True` flag asks the schema
to cast on the way in.

In [ ]:
raw = pd.DataFrame({
    "period":  ["2026Q1", "2026Q1"],
    "region":  ["Europe", "Americas"],
    "product": ["Standard", "Standard"],
    "revenue": ["120000.00", "250000.00"],   # strings
    "units":   ["1200", "2500"],             # strings
})

sales_schema = pa.DataFrameSchema(
    {
        "period":  pa.Column(str),
        "region":  pa.Column(str),
        "product": pa.Column(str),
        "revenue": pa.Column(float),
        "units":   pa.Column(int),
    },
    coerce=True,
)

cleaned = sales_schema.validate(raw)
cleaned.dtypes
# period      object
# region      object
# product     object
# revenue    float64
# units        int64
# dtype: object

`coerce=True` on the schema casts every column. The same flag on a single
`Column` casts only that column, which is useful when one upstream field
is unreliable but the rest are trustworthy.

Coercion runs before checks. A string `"abc"` in a numeric column raises a
coercion error, never a value range error, because the cast itself fails.
This ordering matters when both rules would catch a bad row: the coercion
error is more specific and shows the offending value as the original
string.

Coercion is the right default for ingestion code that reads from Excel,
CSV, or a TM1 view configured to return strings. It is the wrong default
when the schema is being used as a contract between two trusted internal
functions, where a dtype mismatch indicates a bug rather than dirty data
and should fail loudly.

## 7. Nullable, required, and strict mode

A schema describes the columns it cares about. Three flags govern what
happens at the edges of that description: missing values within a column,
missing columns from the frame, and extra columns the frame brings along.

The `nullable=True` flag allows `NaN` (or `None` for object columns). It
is off by default; a column without `nullable=True` rejects any missing
values.

In [ ]:
notes_schema = pa.DataFrameSchema({
    "period": pa.Column(str),
    "notes":  pa.Column(str, nullable=True),    # planners may leave this blank
})

The `required=False` flag allows the column itself to be absent. Useful
for optional measures or commentary fields that some views carry and
others do not.

In [ ]:
forecast_schema = pa.DataFrameSchema({
    "period":   pa.Column(str),
    "revenue":  pa.Column(float),
    "forecast": pa.Column(float, required=False),   # only past current period
})

Strict mode controls extra columns. By default the schema ignores any
column it does not name; `strict=True` rejects them. This catches the case
where an upstream view starts returning a new column that downstream code
mishandles.

In [ ]:
sales_schema = pa.DataFrameSchema(
    {
        "period":  pa.Column(str),
        "region":  pa.Column(str),
        "product": pa.Column(str),
        "revenue": pa.Column(float),
        "units":   pa.Column(int),
    },
    strict=True,
)

with_extra = sales.assign(comments="ok")
sales_schema.validate(with_extra)
# pandera.errors.SchemaError: column 'comments' not in DataFrameSchema

A softer alternative, `strict="filter"`, drops unexpected columns silently
rather than raising. That is the right setting at an ingestion boundary
where the upstream system frequently adds informational columns; raising
strict is right for an internal function whose signature is meant to be
exact.

## 8. Index and MultiIndex schemas

A TM1 cellset is naturally indexed by its dimension tuple, and DataFrames
returned by `execute_view_dataframe` often arrive with a MultiIndex over
the dimensions and value columns for the measures. The schema should
describe that index, not just the columns.

A single index schema declares the index alongside the columns.

In [ ]:
import pandera as pa

indexed_schema = pa.DataFrameSchema(
    columns={
        "revenue": pa.Column(float),
        "units":   pa.Column(int),
    },
    index=pa.Index(str, name="period",
                   checks=pa.Check.str_matches(r"^\d{4}Q[1-4]$")),
)

A MultiIndex schema lists one `Index` per level. The order matters;
pandera matches by position, not name.

In [ ]:
mi_schema = pa.DataFrameSchema(
    columns={
        "revenue": pa.Column(float, pa.Check.ge(0)),
        "units":   pa.Column(int, pa.Check.ge(0)),
    },
    index=pa.MultiIndex([
        pa.Index(str, name="period",  checks=pa.Check.str_matches(r"^\d{4}Q[1-4]$")),
        pa.Index(str, name="region",  checks=pa.Check.isin({"Europe", "Americas", "Asia"})),
        pa.Index(str, name="product", checks=pa.Check.isin({"Standard", "Premium"})),
    ]),
)

sales_indexed = sales.set_index(["period", "region", "product"])
mi_schema.validate(sales_indexed)

Indices support the same dtype and check vocabulary as columns. They also
support `unique=True` (every label is distinct) and `nullable=True`,
though a nullable index is nearly always a sign of a bug upstream and
worth catching rather than allowing.

This is the pattern most useful for cube cells: the dimensions form the
index, the measures form the columns, and the schema declares both. Once
validated, downstream code can rely on `.loc[("2026Q1", "Europe",
"Standard"), "revenue"]` to refer to a unique cell, and on the index labels
to fall within their dimension's element set.

## 9. Lazy validation: collecting every failure

The default `validate(df)` is eager: it raises on the first failure. For a
single error frame that is clear and fast. For a frame with several
independent problems, a wrong region, a negative revenue, a stray column,
eager validation surfaces only the first; the user fixes it, runs again,
surfaces the second, fixes it, and so on. Each cycle pays the cost of
re running the upstream pipeline.

`lazy=True` tells pandera to keep going and report every failure at once.

In [ ]:
broken = sales.copy()
broken.loc[0, "region"]  = "EMEA"        # not in dimension
broken.loc[3, "revenue"] = -50.0         # negative
broken["comments"] = "ok"                # extra column

try:
    sales_schema.validate(broken, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc.failure_cases)
#    schema_context  column   check                              failure_case  index
# 0  DataFrameSchema None     column_in_schema                   comments      None
# 1  Column          region   isin({Americas, Asia, Europe})     EMEA          0
# 2  Column          revenue  greater_than_or_equal_to(0)        -50.0         3

The exception type changes. Eager validation raises `SchemaError`
(singular) carrying details about one failure. Lazy validation raises
`SchemaErrors` (plural) carrying a `failure_cases` DataFrame that
tabulates every problem in one place. Reporting that DataFrame to the
user, or piping it into a CI log, is far more useful than a fix-and-rerun
loop.

Lazy is the right mode for batch jobs, ingestion pipelines, and tests.
Eager is fine in a REPL where each failure is being debugged
interactively. For function decorators (Topic 11), lazy is generally
preferable because the function caller may want to surface the full report
rather than the first error.

## 10. Class-based schemas with DataFrameModel

The `DataFrameSchema({...})` form is direct but quickly becomes unwieldy
as schemas grow and need to be reused. The class based equivalent,
`DataFrameModel`, expresses the same schema using ordinary Python class
syntax with type annotations. This form composes better with type hints,
function decorators (Topic 11), and IDE autocompletion.

In [ ]:
import pandera as pa
from pandera.typing import Series

class SalesSchema(pa.DataFrameModel):
    period:  Series[str]   = pa.Field(str_matches=r"^\d{4}Q[1-4]$")
    region:  Series[str]   = pa.Field(isin={"Europe", "Americas", "Asia"})
    product: Series[str]   = pa.Field(isin={"Standard", "Premium"})
    revenue: Series[float] = pa.Field(ge=0)
    units:   Series[int]   = pa.Field(ge=0)

    class Config:
        strict = True
        coerce = True

SalesSchema.validate(sales)

`pa.Field` carries the same arguments as the keywords on `pa.Column`. The
`Config` inner class carries the schema-level flags (`strict`, `coerce`,
`ordered`, `unique`, etc.) that previously lived on `DataFrameSchema(...)`.

Custom checks attach as decorated methods. `@pa.check` for column-level
checks, `@pa.dataframe_check` for frame-level ones.

In [ ]:
class SalesSchema(pa.DataFrameModel):
    period:  Series[str]   = pa.Field(str_matches=r"^\d{4}Q[1-4]$")
    region:  Series[str]   = pa.Field(isin={"Europe", "Americas", "Asia"})
    product: Series[str]   = pa.Field(isin={"Standard", "Premium"})
    revenue: Series[float] = pa.Field(ge=0)
    units:   Series[int]   = pa.Field(ge=0)

    @pa.dataframe_check
    def premium_min_unit_price(cls, df: pd.DataFrame) -> pd.Series:
        return (df["product"] != "Premium") | (df["revenue"] / df["units"] >= 200)

    class Config:
        strict = True
        coerce = True

Class-based schemas are mostly a question of style; they validate
identically to the `DataFrameSchema({...})` form. The case for them is
reuse: a `DataFrameModel` subclass is itself reusable, importable, and
inheritable. A child schema can extend a parent by adding fields or
tightening checks, mirroring the way regular dataclasses inherit.

## 11. Validating function inputs and outputs

A schema is most valuable at the boundary of a function. Pandera's
`@pa.check_types` decorator pairs a `DataFrameModel` schema with a
function's type annotations. Calling the function with a non conforming
DataFrame raises before the body runs; returning a non conforming
DataFrame raises before the caller sees it.

In [ ]:
import pandera as pa
from pandera.typing import DataFrame, Series

class SalesSchema(pa.DataFrameModel):
    period:  Series[str]   = pa.Field(str_matches=r"^\d{4}Q[1-4]$")
    region:  Series[str]   = pa.Field(isin={"Europe", "Americas", "Asia"})
    product: Series[str]   = pa.Field(isin={"Standard", "Premium"})
    revenue: Series[float] = pa.Field(ge=0)
    units:   Series[int]   = pa.Field(ge=0)

class RegionTotalSchema(pa.DataFrameModel):
    region:  Series[str]   = pa.Field(isin={"Europe", "Americas", "Asia"})
    revenue: Series[float] = pa.Field(ge=0)
    units:   Series[int]   = pa.Field(ge=0)

@pa.check_types
def totals_by_region(sales: DataFrame[SalesSchema]) -> DataFrame[RegionTotalSchema]:
    return (sales
            .groupby("region", as_index=False)
            .agg(revenue=("revenue", "sum"), units=("units", "sum")))

totals_by_region(sales)
#      region   revenue  units
# 0  Americas  520000.0   3580
# 1      Asia  200000.0   1340
# 2    Europe  255000.0   1740

The decorator inspects the type annotations, looks up the matching schema,
and validates on call (input) and on return (output). A function with
multiple DataFrame arguments validates each one against its annotation
independently.

This is the pattern that keeps validation effort proportional to the
number of boundaries, not to the number of operations. A pipeline with
three steps gets three pairs of input/output schemas; the body of each
function trusts its argument and is responsible for producing the declared
output.

For procedural code without type hints, the older `@pa.check_input(schema)`
and `@pa.check_output(schema)` decorators take a schema directly. They are
still supported but rarely the right choice in new code, where
`DataFrameModel` plus `check_types` reads better and integrates with type
checkers.

## 12. Validating against TM1 dimension members

The set of valid values for a dimension column is not a literal. It is
whatever elements the TM1 model currently has under that dimension.
Hardcoding `{"Europe", "Americas", "Asia"}` works on the day the schema is
written and breaks the next time a planner adds a region. Pulling the
allowed set from TM1 at schema build time is the idiomatic fix.

In [ ]:
from TM1py import TM1Service
import pandera as pa

with TM1Service(**credentials) as tm1:
    regions  = set(tm1.elements.get_element_names("Region",  "Region"))
    products = set(tm1.elements.get_element_names("Product", "Product"))
    periods  = set(tm1.elements.get_element_names("Period",  "Period"))

sales_schema = pa.DataFrameSchema({
    "period":  pa.Column(str, pa.Check.isin(periods)),
    "region":  pa.Column(str, pa.Check.isin(regions)),
    "product": pa.Column(str, pa.Check.isin(products)),
    "revenue": pa.Column(float, pa.Check.ge(0)),
    "units":   pa.Column(int, pa.Check.ge(0)),
})

The schema is now a function of the live TM1 model. Two consequences
follow.

First, the schema must be built once per session and reused, not built
inside the validation call. Repeatedly fetching the dimension list per
validation defeats the point and floods the TM1 server with redundant
reads.

Second, the dimension fetch happens against a TM1 connection that is
expected to be open and healthy. If the TM1 service is unreachable, schema
construction fails noisily, which is the right behaviour. A schema that
silently fell back to an empty set would accept every value as out of set
and report every row as failing; one that fell back to a hardcoded set
would accept stale values without notice. Failing at construction makes
the dependency explicit.

For `DataFrameModel`, the same pattern uses `Field(isin=regions)` with
`regions` resolved at class definition time. If the model definition
needs to be loaded before the TM1 connection is open, the dynamic check
can be attached as a `@pa.check` method that closes over a module level
set populated by the connection opening code.

## 13. Synthetic data for tests

A schema is also a generator. `schema.example(size=N)` produces a
DataFrame conforming to every declared dtype, range, set, and pattern,
drawn at random from the legal space.

In [ ]:
sales_schema.example(size=5)
#       period    region   product   revenue  units
# 0     2026Q3    Europe   Standard  44721.18    312
# 1     2027Q1    Asia     Premium     902.50      8
# ...

The generator backs onto the Hypothesis library. Range checks, isin
checks, and string match patterns are translated into Hypothesis
strategies; custom checks are filtered after the fact. The result is a
valid DataFrame that exercises the corners of the allowed space without
anyone writing rows by hand.

The use case is unit testing of downstream functions. A function
`totals_by_region(sales)` can be tested without a live TM1 connection by
feeding it `SalesSchema.example(size=100)` and asserting the output schema
validates the result.

In [ ]:
def test_totals_by_region():
    sample = SalesSchema.example(size=100)
    result = totals_by_region(sample)
    RegionTotalSchema.validate(result)

The synthetic data is not realistic. Region names are random strings from
the legal set, revenues are uniform within range, periods are arbitrary
fiscal quarters. That is fine for shape checking and for catching bugs
that depend on edge values; it is not a substitute for an integration test
against a real TM1 view, which catches dimension churn, dtype drift in the
live cube, and any business rule too subtle to encode as a check.

Synthetic data complements integration tests; it does not replace them.
The two together give fast feedback (synthetic, no network) and ground
truth (integration, real cube).

## 14. Real-world design principles

Once the mechanics are clear, the practical question is when and how to
use them. A handful of guidelines apply to most situations.

**Validate at boundaries, trust internally.** The pandas module made this
point in passing; pandera makes it concrete. The two boundaries that
matter are the moment data enters the process from TM1, a CSV, or another
external source, and the moment data leaves the process toward a TM1 cube
write. Validate aggressively at those two points; do not sprinkle `assert
df.shape[0] > 0` through the body of the pipeline. The value of a schema
is exactly that downstream code does not have to keep checking.

**One schema, two directions.** A function that reads from TM1, summarizes,
and writes back has an input schema (cube cells) and an output schema (the
writeable view shape). Both deserve to exist. The output schema is the
easier one to neglect because it ostensibly comes from your own code; in
practice, the most useful schema is often the one that catches a
regression in your own aggregation logic before the result hits the cube.

**Express domain rules as named checks.** A custom check called
`premium_min_unit_price` is self documenting; an anonymous lambda buried
in a list of checks is not. Names show up in error reports, in failure
logs, in CI dashboards. They are the place future readers will encounter
the rule, often without ever reading the schema definition.

**Prefer DataFrameModel for shared schemas.** A schema used in more than
one place is easier to grow as a class than as a dictionary. Class based
schemas inherit, compose, attach methods, and play well with type hints.
`DataFrameSchema({...})` is fine for a one off validation inside a single
function; `DataFrameModel` is the form that scales to a project.

**Pull dimension lists from TM1, not from constants.** Topic 12 covered the
mechanics. The principle is that the schema should be a derivative of the
live model, not a parallel definition that drifts. The cost is one round
trip on schema construction; the benefit is that a new region added in TM1
is automatically accepted by the schema, and a renamed region is
automatically rejected.

**Lazy in batch, eager in REPL.** Eager validation is fine for an
interactive session where each failure is fixed before the next. Batch
jobs, tests, and ingestion pipelines should use `lazy=True` and surface
every failure in one structured report. Halfway validated data is worse
than unvalidated data because the failure surfaced first may not be the
one that matters.

## 15. Common mistakes

A short collection of errors that are easy to make and worth recognizing
early.

**Using `assert` to validate.** A bare `assert df["revenue"].ge(0).all()`
raises an opaque `AssertionError` and is silently removed when Python runs
with `-O`. Pandera raises a structured exception that names the column,
the check, and the failing rows.

In [ ]:
# Wrong
assert (sales["revenue"] >= 0).all()

# Correct
pa.DataFrameSchema({"revenue": pa.Column(float, pa.Check.ge(0))}).validate(sales)

**Forgetting `coerce` when reading from CSV.** A CSV loaded frame has
`object` dtype on every text looking column, including ones that should
be numeric. The schema rejects it for the wrong dtype before any value
level check runs.

In [ ]:
# Wrong: schema rejects revenue (object dtype) before checking values
sales_schema.validate(pd.read_csv("plan.csv"))

# Correct
sales_schema = pa.DataFrameSchema({...}, coerce=True)
sales_schema.validate(pd.read_csv("plan.csv"))

**Hardcoding dimension element lists.** A schema with a literal
`isin({"Europe", "Americas", "Asia"})` drifts the moment a planner adds a
region. Pull the set from TM1 at schema build time.

In [ ]:
# Wrong
pa.Column(str, pa.Check.isin({"Europe", "Americas", "Asia"}))

# Correct
regions = set(tm1.elements.get_element_names("Region", "Region"))
pa.Column(str, pa.Check.isin(regions))

**Strict mode left off.** Without `strict=True`, an upstream view that
suddenly gains a column passes validation and the new column flows
through, ignored, until something downstream trips on it. Set
`strict=True` on schemas that describe a known shape; use `strict="filter"`
only when extra columns are expected and harmless.

In [ ]:
# Wrong: extra column "comments" passes silently
pa.DataFrameSchema({...}).validate(df_with_extra_column)

# Correct
pa.DataFrameSchema({...}, strict=True).validate(df_with_extra_column)

**Eager validation hiding additional errors.** A frame with five problems
reports one at a time under the default eager mode. Each fix and retry
costs another round trip through the pipeline. `lazy=True` reports them
all.

In [ ]:
# Wrong
sales_schema.validate(broken)   # raises on first error only

# Correct
try:
    sales_schema.validate(broken, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc.failure_cases)    # tabulates every failure

**Anonymous custom checks.** A lambda only check surfaces in error reports
as `<Check _check_fn>`, which gives no information to anyone reading the
failure later. Always name a custom check.

In [ ]:
# Wrong
pa.Check(lambda df: df["revenue"].sum() < 1e8)

# Correct
pa.Check(lambda df: df["revenue"].sum() < 1e8, name="total_cap_100m")

**Rebuilding a TM1 backed schema per call.** When the allowed sets are
fetched from TM1, fetching them inside the validator means every
validation pays for a round trip. Build the schema once, reuse it.

In [ ]:
# Wrong: one TM1 read per validate() call
def validate_sales(df):
    regions = set(tm1.elements.get_element_names("Region", "Region"))
    return pa.DataFrameSchema(
        {"region": pa.Column(str, pa.Check.isin(regions)), ...}
    ).validate(df)

# Correct: build once, reuse
SALES_SCHEMA = pa.DataFrameSchema(
    {"region": pa.Column(str, pa.Check.isin(regions_from_tm1())), ...}
)

def validate_sales(df):
    return SALES_SCHEMA.validate(df)